In [1]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path("/Volumes") / "Crucial X9" # replace with your dataset path

In [2]:
# Display training sessions
import glob
import os

sessions = sorted(glob.glob(os.path.join(DATA_DOWNLOAD_DIR, "emg2pose_dataset_mini/*.hdf5")))
sessions

['/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-11_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-11_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-12_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-12_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-13_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-13_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-16703

In [3]:
# Train model on a subset of sessions (range-based)

import os
import shutil
import subprocess
import pandas as pd
from pathlib import Path

TMP_DIR = DATA_DOWNLOAD_DIR / "emg_sessions"
TMP_DIR.mkdir(exist_ok=True)

# --- parameters ---
start_idx = 0
end_idx = 0  # inclusive

# --- validate + clamp ---
n = len(sessions)
if n == 0:
    raise ValueError("No sessions found")

start_idx = max(0, start_idx)
end_idx = min(n - 1, end_idx)

if start_idx > end_idx:
    raise ValueError("start_idx must be <= end_idx")

selected_sessions = sessions[start_idx:end_idx + 1]
print(f"Training on {len(selected_sessions)} sessions")

# --- clear temp folder ---
for f in TMP_DIR.glob("*"):
    if not f.name.startswith("._"):
        f.unlink()

rows = []

# --- copy sessions + build metadata ---
for session_path in selected_sessions:
    filename = os.path.basename(session_path)
    print("Adding:", filename)

    shutil.copy(session_path, TMP_DIR / filename)

    filename_no_ext = filename.replace(".hdf5", "")
    session_name = filename_no_ext.split("-recording")[0]

    for split in ["train", "val", "test"]:
        rows.append({
            "session": session_name,
            "user": "debug_user",
            "stage": "debug",
            "start": 0,
            "end": 1,
            "side": "right",
            "filename": filename_no_ext,
            "moving_hand": "both",
            "held_out_user": False,
            "held_out_stage": False,
            "split": split,
            "generalization": "debug"
        })

# --- write metadata ---
df = pd.DataFrame(rows)
df.to_csv(TMP_DIR / "metadata.csv", index=False)

print("Created metadata.csv at:", TMP_DIR)



Training on 1 sessions
Adding: 2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_left.hdf5
Created metadata.csv at: /Volumes/Crucial X9/emg_sessions


In [ ]:
# --- run training ONCE on full subset ---
subprocess.run([
    "python", "-m", "emg2pose.train",
    "train=True",
    "eval=True",
    "experiment=tracking_vemg2pose",
    "trainer.max_epochs=100",
    "+callbacks.1.dirpath=/Volumes/Crucial X9/emg_checkpoints",
    f"data_location={TMP_DIR}"
])